# Import libraries

In [ ]:
from experiment_common_code import ExperimentResult, plot_metrics_by_group, plot_confusion_matrix, show_plots, ExperimentResult, save_to_csv
from utils import LANGUAGES, MODEL_NAMES, MODEL_THESIS_NAMES, REVERSE_LABEL_MAP, PROBE_TYPE_FULL_NAME_MAP, get_language_pair_permutations, get_all_language_merged_strings, PROBE_PERFORMANCE_CSVS_FOLDER
from icecream import ic
import pandas as pd
from pandas import DataFrame

import torch as t

from typing import Literal, Any

device: Literal["cuda", "cpu"] = "cuda" if t.cuda.is_available() else "cpu"

# Experiment 1

In [ ]:
!uv run experiment_1.py -pt mm
!uv run experiment_1.py -pt lr

# Japanese original label ablation
!uv run experiment_1.py -pt mm -fol -l jp
!uv run experiment_1.py -pt lr -fol -l jp

# Experiment 2

In [ ]:
!uv run experiment_2.py -pt mm -nr 1
!uv run experiment_2.py -pt lr -nr 2 -ir 2
!uv run experiment_2.py -pt lr -nr 2 -ir 1000

# Japanese original label ablation
!uv run experiment_2.py -pt mm -fol -nr 1
!uv run experiment_2.py -pt lr -fol -nr 2 -ir 2
!uv run experiment_2.py -pt lr -fol -nr 2 -ir 1000

In [ ]:
# Generate the probe_performance CSVs
save_to_csv(
    "test",
    "f1",
    "data/experiment_results/experiment_1",
    probe_type="lr",
    output_folder=PROBE_PERFORMANCE_CSVS_FOLDER,
)
save_to_csv(
    "test",
    "f1",
    "data/experiment_results/experiment_1",
    probe_type="mm",
    output_folder=PROBE_PERFORMANCE_CSVS_FOLDER,
)
save_to_csv(
    "test",
    "marginal_f1",
    "data/experiment_results/experiment_1",
    probe_type="lr",
    output_folder=PROBE_PERFORMANCE_CSVS_FOLDER,
)
save_to_csv(
    "test",
    "marginal_f1",
    "data/experiment_results/experiment_1",
    probe_type="mm",
    output_folder=PROBE_PERFORMANCE_CSVS_FOLDER,
)

save_to_csv(
    "test_a",
    "f1",
    "data/experiment_results/experiment_2",
    probe_type="lr",
    output_folder=PROBE_PERFORMANCE_CSVS_FOLDER,
)
save_to_csv(
    "test_a",
    "f1",
    "data/experiment_results/experiment_2",
    probe_type="mm",
    output_folder=PROBE_PERFORMANCE_CSVS_FOLDER,
)
save_to_csv(
    "test_b",
    "f1",
    "data/experiment_results/experiment_2",
    probe_type="lr",
    output_folder=PROBE_PERFORMANCE_CSVS_FOLDER,
)
save_to_csv(
    "test_b",
    "f1",
    "data/experiment_results/experiment_2",
    probe_type="mm",
    output_folder=PROBE_PERFORMANCE_CSVS_FOLDER,
)

# Experiment 3

Load important variables

In [ ]:
from experiment_3 import run_experiment_3, E3DataframeCreator

model_names: list[str] = MODEL_NAMES
languages: list[str] = LANGUAGES

custom = False
if custom:
    model_names = ["olmo_model"]
    languages = ["en"]
    print(f"Using custom configuration")

ic(custom, model_names, languages)

In [ ]:
run_experiment_3(languages, model_names)
# run_experiment_3(["nl"], ["olmo_model"])
# run_experiment_3(["es", "nl"], ["aya_expanse"])

In [ ]:
dataframe_creator_olmo = E3DataframeCreator("olmo_model", languages, include_strict=False)
df_e3_olmo: DataFrame = dataframe_creator_olmo.create_dataframe()
print(dataframe_creator_olmo.to_latex(df_e3_olmo))
df_e3_olmo

In [ ]:
dataframe_creator_aya_expanse = E3DataframeCreator("aya_expanse", languages, include_strict=False)
df_e3_aya_expanse: DataFrame = dataframe_creator_aya_expanse.create_dataframe()
print(dataframe_creator_aya_expanse.to_latex(df_e3_aya_expanse))
df_e3_aya_expanse

In [ ]:
dataframe_creator_aya_expanse = E3DataframeCreator("aya_expanse", languages, include_strict=True)
df_e3_aya_expanse: DataFrame = dataframe_creator_aya_expanse.create_dataframe()
print(dataframe_creator_aya_expanse.to_latex(df_e3_aya_expanse))
df_e3_aya_expanse

In [ ]:
for model_name in model_names:
    for language in languages:
        exp_result: ExperimentResult = ExperimentResult.get_from_file(3, language, "standard", "model_pred", model_name)
        plot_confusion_matrix(exp_result, "test", include_unknown=True)

# Probe similarity

In [ ]:
# Per-layer experiments (also generates the probe_similarity CSVs in data/csvs/probe_similarity)
!uv run probe_similarity_experiment.py -m olmo_model -t standard control -pt lr -e per_layer_two_metrics -sv
!uv run probe_similarity_experiment.py -m aya_expanse -t standard control -pt lr -e per_layer_two_metrics -sv

# Refitted probe compared with original probe experiments
!uv run probe_similarity_experiment.py -m olmo_model aya_expanse -t standard -pt lr -e per_extra_iter -nr 1 -ir 2 -sv
!uv run probe_similarity_experiment.py -m olmo_model aya_expanse -t standard -pt lr -e per_extra_iter -nr 1 -ir 1000 -sv

# Plots

In [ ]:
for probe_type in ["lr", "mm"]:
        show_plots(1, 
                MODEL_NAMES, 
                ["test"], 
                LANGUAGES, 
                ["standard"], 
                metric="f1",
                probe_type=probe_type,
                legend_position="upper left",
                subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
                y_axis_range=(0.45, 1.0),
                show=True,
                save=True,
                filename=f"e1_f1_{probe_type}.png",
                as_bars=False,
                )

In [ ]:
for probe_type in ["lr", "mm"]:
        show_plots(1, 
                MODEL_NAMES, 
                ["test"], 
                LANGUAGES, 
                ["standard"], 
                metric="f1",
                probe_type=probe_type,
                legend_position="upper left",
                subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
                y_axis_range=(0.45, 1.0),
                show=False,
                as_bars=False,
                print_lines=True,
                layer_to_print=-1
                )

In [ ]:
for probe_type in ["lr", "mm"]:
    show_plots(1, 
                    MODEL_NAMES, 
                    ["test"], 
                    LANGUAGES, 
                    ["standard"], 
                    metric="marginal_f1",
                    probe_type=probe_type,
                    horizontal_line=0,
                    legend_position="upper left",
                    subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
                    y_axis_range=(-0.3, 0.4),
                    show=False,
                    as_bars=False,
                    print_lines=True
                    )

In [ ]:
for probe_type in ["lr", "mm"]:
        show_plots(1, 
                MODEL_NAMES, 
                ["test"], 
                LANGUAGES, 
                ["standard"], 
                metric="marginal_f1",
                probe_type=probe_type,
                horizontal_line=0,
                legend_position="upper left",
                subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
                y_axis_range=(-0.3, 0.4),
                show=True,
                save=True,
                filename=f"e1_selectivity_{probe_type}.png",
                as_bars=False,
                )

In [ ]:
def make_experiment_2_plot(model_name, probe_type, extra_iter_num, legend_position="upper left", language_pairs: list[tuple[str, str]]|None=None, filename_suffix="", as_bars=True) -> None:
    filename = f"e2_{probe_type}_{extra_iter_num}_{model_name}{'_ablation' if language_pairs is not None else ''}{'_'+filename_suffix if filename_suffix else ''}"

    if language_pairs is None:
        language_pairs = get_language_pair_permutations(LANGUAGES)
    
    print(language_pairs)
    
    language_pairs_as_strings: list[str] = get_all_language_merged_strings(language_pairs)
    
    print(f"Creating plot of {model_name} {probe_type} with {extra_iter_num} extra iters")
    show_plots(2, 
                [model_name], 
                ["test_b", "test_a"],
                language_pairs_as_strings,
                ["standard"], 
                [extra_iter_num],
                metric="f1",
                separate_chars_within_plot=["language_b", "split", "probing_task"],
                probe_type=probe_type,
                horizontal_line="baseline_f1",
                legend_position=legend_position,
                subplot_titles=[],
                y_axis_range=(0.0, 1.0),
                save=True,
                show=True,
                filename=filename,
                as_bars=as_bars
                )

In [ ]:
# 0 extra iters
for probe_type in ["lr", "mm"]:
    for model_name in MODEL_NAMES:
        make_experiment_2_plot(model_name, probe_type, 0)

In [ ]:
#  2 extra iters
for model_name in MODEL_NAMES:
    make_experiment_2_plot(model_name, "lr", 2, legend_position="lower center")

In [ ]:
#  1000 extra iters
for model_name in MODEL_NAMES:
    make_experiment_2_plot(model_name, "lr", 1000, legend_position="lower center")

In [ ]:
# Ablation: Japanese with original labels
probe_type = "lr"
# for probe_type in ["lr", "mm"]:
show_plots(1, 
        MODEL_NAMES, 
        ["test"], 
        ["jp", "jp_original_labels"], 
        ["standard"], 
        metric="f1",
        probe_type=probe_type,
        legend_position="upper left",
        subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
        y_axis_range=(0.3, 1.0),
        show=True,
        save=True,
        filename=f"e1_f1_{probe_type}_jp_ablation.png",
        as_bars=False,
        )

show_plots(1, 
        MODEL_NAMES, 
        ["test"], 
        ["jp", "jp_original_labels"], 
        ["standard"], 
        metric="marginal_f1",
        probe_type=probe_type,
        horizontal_line=0,
        legend_position="upper left",
        subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
        y_axis_range=(-0.3, 0.6),
        show=True,
        save=True,
        filename=f"e1_selectivity_{probe_type}_jp_ablation.png",
        as_bars=False,
        )

In [ ]:
probe_type="lr"
# for probe_type in ["lr", "mm"]:
for model_name in MODEL_NAMES:
    make_experiment_2_plot(model_name, probe_type, 0, language_pairs=[
        ("en", "jp"), ("es", "jp"), ("nl", "jp"),
        ("en", "jp_original_labels"), ("es", "jp_original_labels"), ("nl", "jp_original_labels"),
    ],filename_suffix="1")

    make_experiment_2_plot(model_name, probe_type, 0, language_pairs=[
        ("jp", "en"), ("jp", "es"), ("jp", "nl"),
        ("jp_original_labels", "en"), ("jp_original_labels", "es"), ("jp_original_labels", "nl"),
    ],filename_suffix="2")

In [ ]:
# line plot ablations
for model_name in MODEL_NAMES:
        make_experiment_2_plot(model_name, "lr", 0, as_bars=False)
        make_experiment_2_plot(model_name, "lr", 2, as_bars=False)
        make_experiment_2_plot(model_name, "lr", 1000, as_bars=False)